# 08 · ACDC 무증강: Swin scratch / VoCo + 기존 nnU-Net 재평가

**각 작업 독립 실행 · 서버 전용.** 기존 `acdc-data`를 repo와 같은 부모 폴더에 복사하세요. 준비된 데이터가 없으면 ACDC 01을 먼저 실행합니다.

- 기존 nnU-Net `none`: **재학습 없음**, DS ON checkpoint의 final/best를 mirror TTA OFF로 재평가.
- Swin UNETR v2 scratch / VoCo: 두 실험만 새로 학습. DS OFF, 증강 전부 OFF, AMP ON.
- fold 0, 250 epochs, train 250 / val 50 step, batch 4, 패치 `[20,256,224]`, 기존 전처리·SGD·Dice+CE 유지.
- VoCo 공식 **Base SSL, PreCT-160K 확장판** 가중치 사용. CVPR 원본 checkpoint와 구분합니다.
- 두 Swin 모두 feature48. `swinViT` encoder 전체만 이전하고 decoder/output은 동일 seed 무작위 초기화. 전체 모델을 미세조정합니다.
- 기존 nnU-Net은 DS ON·과거 seed, Swin은 DS OFF·고정 seed라는 차이가 남습니다. 단일 fold/seed 탐색 실험입니다.
- 최종 전체 영상 Dice/mIoU는 모두 같은 코드로 계산합니다. **final 주 결과, best 보조 결과**. 기존 TTA 결과는 비교에 사용하지 않습니다.

실행 순서: **공통 1~3절 → 원하는 작업 셀 하나(4A / 4B / 4C) → 모니터링**.
다른 인스턴스에서는 4A·4B를 건너뛰고 **4C만 실행**할 수 있습니다.
Swin 실험 서버에는 다른 모델의 체크포인트가 필요 없습니다. 준비된 ACDC 데이터와 같은 split/plans를 복사하세요.
기존 nnU-Net 체크포인트는 **4A를 실행할 인스턴스에만** 필요합니다. VoCo 가중치는 4C에서만 다운로드합니다.


## 1. 초기화 및 경로 — Python 3.11 venv 커널 선택

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util, shlex
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'nnunetv2').is_dir() and (p/'pyproject.toml').is_file()), None)
if REPO is None:
    raise RuntimeError('nnUNet 저장소 안에서 노트북을 여세요.')
EXPERIMENT = REPO/'experiments/acdc'
TOOLS = EXPERIMENT/'pretraining_comparison'
STORAGE_ROOT = REPO.parent/'acdc-data'
BASELINE_RUN_ID = 'acdc-aug250-v1-none'
SUITE_ID = 'acdc-noaug-swin-voco250-v1'
GPU = '0'
SEED = 12345
AUGMENTATION_WORKERS = 4  # 증강 OFF여도 패치 로더 worker는 필요합니다.
INSTALL_PACKAGES = False
if sys.platform != 'linux':
    raise RuntimeError('이 노트북은 Linux GPU 서버에서만 실행하세요.')
if sys.version_info[:2] != (3,11) or sys.prefix == sys.base_prefix:
    raise RuntimeError('Python 3.11 venv 커널을 선택하세요.')
for f in [TOOLS/'runner.py', TOOLS/'requirements.txt', TOOLS/'download_weights.py']:
    if not f.is_file(): raise FileNotFoundError(f'노트북만 복사하지 말고 저장소 전체를 업데이트하세요: {f}')
print('Kernel:', sys.executable)
print('Data:', STORAGE_ROOT)
print('Existing baseline:', STORAGE_ROOT/'runs'/BASELINE_RUN_ID)


## 2. 의존성 설치 — 빈 venv라면 INSTALL_PACKAGES=True 후 실행
기존 GPU용 torch가 있으면 버전을 보존합니다. 새 환경의 기본 CUDA wheel은 cu126입니다. 설치 후 커널을 재시작하고 설치 스위치를 False로 되돌리세요.

In [ ]:
CUDA_WHEEL = 'cu126'  # 새 torch 설치 시 서버 NVIDIA driver와 맞추세요.
if INSTALL_PACKAGES:
    import importlib.metadata
    pip = [sys.executable, '-m', 'pip']
    if importlib.util.find_spec('torch') is None:
        subprocess.run(pip+['install','--no-cache-dir','torch==2.8.0',
                           '--index-url',f'https://download.pytorch.org/whl/{CUDA_WHEEL}'],check=True)
    torch_pin = 'torch==' + importlib.metadata.version('torch')
    subprocess.run(pip+['install','--no-cache-dir',torch_pin,'-e',str(REPO),
                       '-r',str(EXPERIMENT/'requirements-server.txt'),
                       '-r',str(TOOLS/'requirements.txt')],check=True)
    subprocess.run(pip+['check'],check=True)
    print('설치 완료: 커널 재시작 후 1번부터 실행하세요. INSTALL_PACKAGES=False.')
else:
    print('설치 생략')


## 3. 공통 설정 및 독립 실행 함수
이 셀은 학습·추론·가중치 다운로드를 시작하지 않습니다. 모든 인스턴스에서 동일한 SUITE_ID/seed와 기존 ACDC 전처리·분할을 사용하세요. 실행 경로와 GPU 번호는 인스턴스별로 달라도 됩니다.

In [ ]:
sys.path.insert(0,str(TOOLS))
sys.path.insert(0,str(EXPERIMENT/'scripts'))
spec = importlib.util.spec_from_file_location('acdc_pretraining_runner', TOOLS/'runner.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
suite = dict(suite_id=SUITE_ID, storage_root=str(STORAGE_ROOT.resolve()),
    baseline_run_id=BASELINE_RUN_ID, execution_enabled=True,
    expected_git_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(),
    gpu=GPU, augmentation_workers=AUGMENTATION_WORKERS, models=['scratch','voco'],
    fold=0, epochs=250, batch_size=4, seed=SEED, evaluate_best=True)
runner.validate_suite(suite)
SUITE_DIR = runner.suite_dir(suite)

def launch_task(task, start=False, resume=False):
    import shutil
    if not start:
        print(task, ': 실행 스위치 False')
        return
    if not shutil.which('tmux'):
        raise RuntimeError('서버에 tmux를 설치하세요: apt-get install -y tmux')
    session = 'acdc-' + SUITE_ID + '-' + task
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode == 0:
        print('이미 실행 중:',session)
        return
    config = dict(suite)
    if task == 'voco':
        import download_weights
        checkpoint, meta = download_weights.download(STORAGE_ROOT/'pretrained')
        config.update(voco_checkpoint=str(checkpoint.resolve()),voco_sha256=meta['sha256'],voco_source=meta)
    folder = runner.task_dir(config,task)
    folder.mkdir(parents=True,exist_ok=True)
    config_path = folder/'config.json'
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise ValueError('해당 작업 설정이 바뀌었습니다. 새 SUITE_ID를 사용하세요.')
    runner.write_json(config_path,config)
    command=[sys.executable,'-u','-B',str(TOOLS/'runner.py'),'--config',str(config_path),'--task',task]
    if resume: command.append('--resume')
    shell_command='exec '+shlex.join(command)+' >> '+shlex.quote(str(folder/'runner.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,shell_command],
        env=runner.environment(runner.model_config(config,'scratch')),cwd=REPO,check=True)
    print('시작:',session)
    print('로그:',folder/'runner.log')

print('공통 설정 준비 완료. 4A/4B/4C 중 원하는 셀만 실행하세요.')


## 4A. 기존 nnU-Net 무증강 모델 재평가
기존 checkpoint final/best가 필요합니다. 재학습 없이 TTA OFF로 추론합니다.
실행하려면 이 셀의 START 스위치를 True로 변경하세요. 같은 GPU에서는 동시에 실행하지 마세요. 재개할 때 해당 RESUME만 True로 설정합니다.

In [ ]:
START_UNET = False
RESUME_UNET = False
launch_task('unet_existing', start=START_UNET, resume=RESUME_UNET)


## 4B. Swin 무작위 초기화 학습
4A를 실행할 필요가 없습니다. 기존 U-Net checkpoint와 VoCo 가중치도 필요 없습니다.
실행하려면 이 셀의 START 스위치를 True로 변경하세요. 같은 GPU에서는 동시에 실행하지 마세요. 재개할 때 해당 RESUME만 True로 설정합니다.

In [ ]:
START_SCRATCH = False
RESUME_SCRATCH = False
launch_task('scratch', start=START_SCRATCH, resume=RESUME_SCRATCH)


## 4C. VoCo 초기화 학습 — 다른 인스턴스에서 단독 실행 가능
공통 1~3절 실행 후 이 셀로 바로 오세요. 4A/4B 결과는 필요 없습니다. 공식 가중치 약 220 MB만 이때 다운로드합니다.
실행하려면 이 셀의 START 스위치를 True로 변경하세요. 같은 GPU에서는 동시에 실행하지 마세요. 재개할 때 해당 RESUME만 True로 설정합니다.

In [ ]:
START_VOCO = False
RESUME_VOCO = False
launch_task('voco', start=START_VOCO, resume=RESUME_VOCO)


## 5. 작업별 상태·loss 모니터링
해당 서버에 없는 작업은 미실행으로 표시합니다. 이 셀은 학습을 시작하지 않습니다.

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from workflow import parse_losses
for task in ['unet_existing','scratch','voco']:
    folder=runner.task_dir(suite,task)
    state=folder/'status.json'
    print(task, ':', state.read_text() if state.exists() else '이 인스턴스에서 미실행')
    log=folder/'runner.log'
    if log.exists(): print('\n'.join(log.read_text(errors='replace').splitlines()[-12:]))
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,name in zip(axes,['scratch','voco']):
    # Monitoring paths do not require VoCo weights on this instance.
    config=dict(suite,voco_sha256=None)
    folder=runner.model_dir(runner.model_config(config,name))/'fold_0'
    rows=parse_losses(folder)
    if rows:
        frame=pd.DataFrame(rows)
        ax.plot(frame.epoch,frame.train_loss,label='train')
        ax.plot(frame.epoch,frame.val_loss,label='validation')
        ax.legend()
    ax.set(title=name,xlabel='Epoch (0-based)',ylabel='Dice + CE loss')
plt.tight_layout()
plt.show()


## 6. 인스턴스별 결과 합치기·비교
다른 서버의 `pretraining_suites/<SUITE_ID>/evaluation_no_tta/`를 분석 서버의 별도 폴더에 복사하세요.
비교만 하려면 각 모델/final·best의 **summary.json**만 있어도 됩니다. manifest.json·case-metrics.csv도 함께 보관하는 것을 권장합니다. 큰 예측 NIfTI나 checkpoint는 비교표 작성에 필요 없습니다.
`RESULT_ROOTS`에는 `evaluation_no_tta`를 포함한 **부모 폴더**를 나열합니다. 모델 일부만 있어도 부분 결과를 표시합니다.
경로/GPU가 달라도 비교 가능하며 split/plans/평가 코드/핵심 설정이 다르면 병합을 중단합니다. 중복 모델 결과가 다를 때도 중단합니다.


In [ ]:
RESULT_ROOTS = [SUITE_DIR]
# 예: RESULT_ROOTS += [STORAGE_ROOT/'imported_results'/'server_voco']
rows=runner.write_comparison(suite,roots=RESULT_ROOTS)
if rows:
    table=pd.DataFrame(rows)
    display(table.drop(columns=['comparison_protocol'],errors='ignore'))
    fig,ax=plt.subplots(figsize=(9,4))
    table.pivot(index='model',columns='checkpoint',values='dice_percent').plot.bar(ax=ax)
    ax.set(ylabel='Mean Dice (%)',title='ACDC — no augmentation / no TTA; U-Net DS ON / Swin DS OFF')
    ax.tick_params(axis='x',rotation=0)
    fig.tight_layout()
    fig.savefig(SUITE_DIR/'comparison.png',dpi=180)
    plt.show()
    print('저장 위치:',SUITE_DIR)
else:
    print('아직 평가 결과가 없습니다. 작업별 로그 또는 RESULT_ROOTS를 확인하세요.')


## 참고
- [VoCo 공식 확장판](https://github.com/Luffy03/Large-Scale-Medical), [가중치](https://huggingface.co/Luffy503/VoCo).
- batch4 GPU smoke는 선택한 Swin 모델만 검사합니다. OOM 시 자동으로 batch/patch를 바꾸지 않습니다.
- 기존 U-Net은 DS ON·과거 seed, 새 Swin은 DS OFF·고정 seed라는 차이가 있습니다.
- 작업마다 독립된 config/snapshot/status/log를 저장합니다. 한 작업의 실패가 다른 작업을 시작시키지 않습니다.
